# Marks reader: TrOCR run (section A, the ResNet run, is already done)
Kaggle settings: **GPU (T4/P100), Internet ON**. Run the cells below in order: 1 setup, 2 install, then **restart the session once**, 3 setup again, 4 TrOCR, 5 results.

Baseline to beat (current model, held-out rows): **89% overall, 98% single-number rows, 46% sum rows.**

### 1. Setup (prints `True True True`)

In [ ]:
import glob, os, shutil, subprocess
os.makedirs('/kaggle/working/proj', exist_ok=True)
# Kaggle usually auto-extracts an uploaded zip, so look for the extracted folder first, then for the zip itself
hits = glob.glob('/kaggle/input/**/scripts/train_marks_sum.py', recursive=True)
if hits:
    src = os.path.dirname(os.path.dirname(hits[0]))
    shutil.copytree(src, '/kaggle/working/proj', dirs_exist_ok=True)
    print('copied extracted dataset from', src)
else:
    z = glob.glob('/kaggle/input/**/kaggle_marks.zip', recursive=True) + glob.glob('/kaggle/working/kaggle_marks.zip')
    assert z, 'Dataset not found: use "Add Input" on the right and attach your marks dataset, then re-run this cell.'
    subprocess.run(['unzip', '-qo', z[0], '-d', '/kaggle/working/proj'], check=True)
    print('unzipped', z[0])
os.chdir('/kaggle/working/proj')
print(os.path.exists('scripts/train_marks_trocr.py'), os.path.exists('data/cells.npz'), os.path.exists('models/marks.pt'))

### 2. Install the tokenizer packages, then **Run > Restart session**, then re-run cell 1 (setup) and carry on from cell 4

In [ ]:
!pip -q install sentencepiece protobuf

### 4. TrOCR-small-handwritten, fine-tuned on the same rows (25 epochs)

In [ ]:
!python -u scripts/train_marks_trocr.py --epochs 25 --mnist 400 --out models/marks_trocr 2>&1 | grep -v -i 'warn\|Loading weights' | tee trocr.log

### 5. Results (paste this output to Claude, together with the ResNet one from section A)

In [ ]:
!grep -A12 'VALIDATION' trocr.log
!ls models/marks_trocr

### 6. Download
Zip the TrOCR model only if it beats the ResNet run:

In [ ]:
!cd models && zip -qr marks_trocr.zip marks_trocr && ls -la marks_trocr.zip

---
### Optional: section A again (ResNet-CTC + MNIST), already done


In [ ]:
# !python -u scripts/train_marks_sum.py --epochs 120 --mnist 400 --out models/marks_sum_mnist.pt 2>&1 | grep -v -i warn | tee resnet_mnist.log